# COVID-19 Mortality & Risk Factor Analysis — Mexico

### Executive Summary
This notebook processes and integrates two major epidemiological datasets from the Mexican Ministry of Health (**1.1M+ patient records**). The pipeline focuses on data harmonization, null value treatment, and feature engineering to evaluate Case Fatality Rates (CFR) across demographic segments and comorbidity profiles.

**Key Objectives:**
1. Clean and integrate heterogeneous historical and recent datasets.
2. Optimize memory usage for million-row processing.
3. Export structured `csv` artifacts to fuel a Power BI star schema.

In [60]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Directory setup — works when the notebook runs from notebooks/ or project root
def resolve_project_root() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "data" / "raw" / "COVID19MEXICO.csv").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Mapping variables (SSA / INEGI Official Codes)
STATE_MAPPING = {
    1: "AGUASCALIENTES", 2: "BAJA CALIFORNIA", 3: "BAJA CALIFORNIA SUR",
    4: "CAMPECHE", 5: "COAHUILA DE ZARAGOZA", 6: "COLIMA",
    7: "CHIAPAS", 8: "CHIHUAHUA", 9: "CIUDAD DE MÉXICO",
    10: "DURANGO", 11: "GUANAJUATO", 12: "GUERRERO",
    13: "HIDALGO", 14: "JALISCO", 15: "MÉXICO",
    16: "MICHOACÁN DE OCAMPO", 17: "MORELOS", 18: "NAYARIT",
    19: "NUEVO LEÓN", 20: "OAXACA", 21: "PUEBLA",
    22: "QUERÉTARO", 23: "QUINTANA ROO", 24: "SAN LUIS POTOSÍ",
    25: "SINALOA", 26: "SONORA", 27: "TABASCO",
    28: "TAMAULIPAS", 29: "TLAXCALA", 30: "VERACRUZ DE IGNACIO DE LA LLAVE",
    31: "YUCATÁN", 32: "ZACATECAS", 99: "NOT_SPECIFIED",
}

INEGI_TO_ISO_MAPPING = {
    f"{i:02d}": f"MX{code}" for i, code in zip(range(1, 33), [
        "AGS", "BCN", "BCS", "CAM", "COA", "COL", "CHP", "CHH", "MEX", "DUR",
        "GUA", "GRO", "HID", "JAL", "MEX", "MIC", "MOR", "NAY", "NLE", "OAX",
        "PUE", "QUE", "ROO", "SLP", "SIN", "SON", "TAB", "TAM", "TLA", "VER",
        "YUC", "ZAC"
    ])
}

# Column Translation Dictionary 
# Note: Keeps source values mapped to clean English column names
COLUMN_TRANSLATION = {
    "USMER": "ORIGIN", "MEDICAL_UNIT": "SECTOR", "SEX": "SEX",
    "PATIENT_TYPE": "PATIENT_TYPE", "DATE_DIED": "DEATH_DATE",
    "INTUBED": "INTUBATED", "PNEUMONIA": "PNEUMONIA", "AGE": "AGE",
    "PREGNANT": "PREGNANT", "DIABETES": "DIABETES", "COPD": "COPD",
    "ASTHMA": "ASTHMA", "INMSUPR": "IMMUNOSUPPRESSED", "HIPERTENSION": "HYPERTENSION",
    "OTHER_DISEASE": "OTHER_COMORBIDITY", "CARDIOVASCULAR": "CARDIOVASCULAR",
    "OBESITY": "OBESITY", "RENAL_CHRONIC": "CHRONIC_RENAL",
    "TOBACCO": "TOBACCO_USE", "ICU": "ICU", "CLASIFICATION_FINAL": "FINAL_COVID_CLASSIFICATION",
}

# Translation dictionary for the Spanish Dataset (df1)
SPANISH_COLUMN_TRANSLATION = {
    "FECHA_ACTUALIZACION": "LAST_UPDATED",
    "ID_REGISTRO": "REGISTRATION_ID",
    "ORIGEN": "ORIGIN",
    "SECTOR": "SECTOR",
    "ENTIDAD_UM": "ENTIDAD_UM",           # Kept as requested
    "SEXO": "SEX",
    "ENTIDAD_NAC": "ENTIDAD_NAC",         # Kept as requested
    "ENTIDAD_RES": "ENTIDAD_RES",         # Kept as requested
    "MUNICIPIO_RES": "MUNICIPIO_RES",     # Kept as requested
    "TIPO_PACIENTE": "PATIENT_TYPE",
    "FECHA_INGRESO": "ADMISSION_DATE",
    "FECHA_SINTOMAS": "SYMPTOMS_ONSET_DATE",
    "FECHA_DEF": "DEATH_DATE",
    "INTUBADO": "INTUBATED",
    "NEUMONIA": "PNEUMONIA",
    "EDAD": "AGE",
    "NACIONALIDAD": "NATIONALITY",
    "EMBARAZO": "PREGNANT",
    "HABLA_LENGUA_INDIG": "SPEAKS_INDIGENOUS_LANGUAGE",
    "INDIGENA": "INDIGENOUS",
    "DIABETES": "DIABETES",
    "EPOC": "COPD",
    "ASMA": "ASTHMA",
    "INMUSUPR": "IMMUNOSUPPRESSED",
    "HIPERTENSION": "HYPERTENSION",
    "OTRA_COM": "OTHER_COMORBIDITY",
    "CARDIOVASCULAR": "CARDIOVASCULAR",
    "OBESIDAD": "OBESITY", 
    "RENAL_CRONICA": "CHRONIC_RENAL",
    "TABAQUISMO": "TOBACCO_USE",
    "OTRO_CASO": "CONTACT_WITH_OTHER_CASE",
    "TOMA_MUESTRA_LAB": "LAB_SAMPLE_TAKEN",
    "RESULTADO_PCR": "PCR_RESULT",
    "RESULTADO_PCR_COINFECCION": "PCR_COINFECTION_RESULT",
    "TOMA_MUESTRA_ANTIGENO": "ANTIGEN_SAMPLE_TAKEN",
    "RESULTADO_ANTIGENO": "ANTIGEN_RESULT",
    "CLASIFICACION_FINAL_COVID": "FINAL_COVID_CLASSIFICATION",
    "CLASIFICACION_FINAL_FLU": "FINAL_FLU_CLASSIFICATION",
    "MIGRANTE": "MIGRANT",
    "PAIS_NACIONALIDAD": "COUNTRY_OF_NATIONALITY",
    "PAIS_ORIGEN": "COUNTRY_OF_ORIGIN",
    "UCI": "ICU"
}

BINARY_COLUMNS = [
    "DIABETES", "COPD", "ASTHMA", "IMMUNOSUPPRESSED", "HYPERTENSION", "OTHER_COMORBIDITY",
    "CARDIOVASCULAR", "OBESITY", "CHRONIC_RENAL", "TOBACCO_USE",
    "INTUBATED", "PNEUMONIA", "ICU", "PREGNANT",
]

COMORBIDITIES_FOR_ANALYSIS = [
    "DIABETES", "HYPERTENSION", "OBESITY", "COPD", "ASTHMA",
    "CHRONIC_RENAL", "CARDIOVASCULAR", "TOBACCO_USE", "IMMUNOSUPPRESSED", "OTHER_COMORBIDITY",
]

COMORBIDITY_COLUMNS = [
    "DIABETES", "COPD", "ASTHMA", "IMMUNOSUPPRESSED", "HYPERTENSION", "OTHER_COMORBIDITY",
    "CARDIOVASCULAR", "OBESITY", "CHRONIC_RENAL", "TOBACCO_USE",
]


def export_processed_csv(df: pd.DataFrame, file_name: str) -> None:
    # Exports refined datasets to the processed data directory for Power BI ingestion.
    output_path = PROCESSED_DIR / file_name
    df.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Exported artifact: {output_path}")

print("Configuration loaded.")
print(f"  Project root : {PROJECT_ROOT}")
print(f"  Raw data dir : {RAW_DIR}")
print(f"  Output dir   : {PROCESSED_DIR}")
print(f"  COVID19 CSV  : {(RAW_DIR / 'COVID19MEXICO.csv').exists()}")
print(f"  Covid Data   : {(RAW_DIR / 'Covid Data.csv').exists()}")

Configuration loaded.
  Project root : c:\Users\willi\Documents\3 cuatri Datos\DataProcessing_Proyect
  Raw data dir : c:\Users\willi\Documents\3 cuatri Datos\DataProcessing_Proyect\data\raw
  Output dir   : c:\Users\willi\Documents\3 cuatri Datos\DataProcessing_Proyect\data\processed
  COVID19 CSV  : True
  Covid Data   : True


# 1. Data Loading & Memory Optimization

In [61]:
# 1. Data Loading & Memory Optimization
optimal_dtypes = {
    'SECTOR': 'category',
    'ENTIDAD_UM': 'category',
    'ENTIDAD_NAC': 'category',
    'ENTIDAD_RES': 'category',
    'MUNICIPIO_RES': 'category',
    'TIPO_PACIENTE': 'category',
    'NACIONALIDAD': 'category'
}

print("Ingesting datasets...")
# Load df1 (Spanish) and df2 (English)
df1 = pd.read_csv(RAW_DIR / "COVID19MEXICO.csv", dtype=optimal_dtypes, low_memory=False)
df2 = pd.read_csv(RAW_DIR / "Covid Data.csv", low_memory=False)

print(f"Dataset 1 (Recent) loaded: {len(df1):,} rows")
print(f"Dataset 2 (Historical) loaded: {len(df2):,} rows")

Ingesting datasets...
Dataset 1 (Recent) loaded: 137,030 rows
Dataset 2 (Historical) loaded: 1,048,575 rows


In [62]:
df2.head()

,USMER,MEDICAL_UNIT,SEX,PATIENT_TYPE,DATE_DIED,INTUBED,PNEUMONIA,AGE,PREGNANT,DIABETES,...,ASTHMA,INMSUPR,HIPERTENSION,OTHER_DISEASE,CARDIOVASCULAR,OBESITY,RENAL_CHRONIC,TOBACCO,CLASIFFICATION_FINAL,ICU
0,2,1,1,1,03/05/2020,97,1,65,2,2,...,2,2,1,2,2,2,2,2,3,97
1,2,1,2,1,03/06/2020,97,1,72,97,2,...,2,2,1,2,2,1,1,2,5,97
2,2,1,2,2,09/06/2020,1,2,55,97,1,...,2,2,2,2,2,2,2,2,3,2
3,2,1,1,1,12/06/2020,97,2,53,2,2,...,2,2,2,2,2,2,2,2,7,97
4,2,1,2,1,21/06/2020,97,2,68,97,1,...,2,2,1,2,2,2,2,2,3,97


## 2. Harmonization, Cleaning, and Unification

In [63]:
# Rename df1 using the Spanish-to-English map
df1 = df1.rename(columns=SPANISH_COLUMN_TRANSLATION)

# Rename df2 using your original English-to-Clean-English map (COLUMN_TRANSLATION)
df2 = df2.rename(columns=COLUMN_TRANSLATION)

print("Columns successfully standardized. Ready for merging")

Columns successfully standardized. Ready for merging


In [64]:
print(f"Codes in df1 for DIABETES: {df1['DIABETES'].unique()}")
print(f"Codes in df2 for DIABETES: {df2['DIABETES'].unique()}")

for df in (df1, df2):
    for col in BINARY_COLUMNS:
        if col in df.columns:
            df[col] = df[col].replace({98: np.nan})
            df[col] = pd.to_numeric(df[col], errors="coerce")

print("Cleanup of code 98 values completed.")

Codes in df1 for DIABETES: [ 2  1 98]
Codes in df2 for DIABETES: [ 2  1 98]
Cleanup of code 98 values completed.


In [65]:
df1_columns = df1.columns.tolist()
df2_aligned = df2.reindex(columns=df1_columns)
df_unified = pd.concat([df1, df2_aligned], ignore_index=True)

print(f"df1 records: {len(df1):,}")
print(f"df2 records: {len(df2):,}")
print(f"Total unified records: {len(df_unified):,}")
print(f"Columns: {df_unified.shape[1]}")
df_unified.head()

df1 records: 137,030
df2 records: 1,048,575
Total unified records: 1,185,605
Columns: 42


,LAST_UPDATED,REGISTRATION_ID,ORIGIN,SECTOR,ENTIDAD_UM,SEX,ENTIDAD_NAC,ENTIDAD_RES,MUNICIPIO_RES,PATIENT_TYPE,...,PCR_RESULT,PCR_COINFECTION_RESULT,ANTIGEN_SAMPLE_TAKEN,ANTIGEN_RESULT,FINAL_COVID_CLASSIFICATION,FINAL_FLU_CLASSIFICATION,MIGRANT,COUNTRY_OF_NATIONALITY,COUNTRY_OF_ORIGIN,ICU
0,2025-11-18,g771db5,1,15,21,1,21,21,085,1,...,997.0,997.0,2.0,97.0,6.0,6.0,99.0,México,97,97
1,2025-11-18,g574d0a,1,4,09,1,09,09,017,1,...,5.0,5.0,2.0,97.0,7.0,7.0,99.0,México,97,97
2,2025-11-18,g941a21,1,12,10,1,10,10,005,1,...,3.0,998.0,2.0,97.0,7.0,3.0,99.0,México,97,97
3,2025-11-18,geaaed1,1,4,09,2,09,09,006,2,...,5.0,5.0,2.0,97.0,7.0,7.0,99.0,México,97,2
4,2025-11-18,g9c2f41,1,5,24,2,24,24,014,2,...,17.0,17.0,2.0,97.0,5.0,5.0,99.0,México,97,2


## 3. Feature Engineering (Derived Variables)

In [66]:
df_unified["DEATH_DATE"] = pd.to_datetime(df_unified["DEATH_DATE"], errors="coerce")
df_unified["DECEASED"] = df_unified["DEATH_DATE"].notna().astype(int)

print("Death count analysis:")
print(df_unified["DECEASED"].value_counts())

Death count analysis:
DECEASED
0    1103573
1      82032
Name: count, dtype: int64


In [67]:
df_unified["AGE"] = pd.to_numeric(df_unified["AGE"], errors="coerce")

age_bins = [0, 18, 36, 60, df_unified["AGE"].max() + 1]
age_labels = ["0-17", "18-35", "36-59", "60+"]

df_unified["AGE_GROUP"] = pd.cut(
    df_unified["AGE"],
    bins=age_bins,
    labels=age_labels,
    right=False,
    include_lowest=True,
)

print("Value counts by age group:")
print(df_unified["AGE_GROUP"].value_counts(dropna=False))

Value counts by age group:
AGE_GROUP
36-59    521298
18-35    382854
60+      186273
0-17      95180
Name: count, dtype: int64


## 4. Case Fatality Rate (CFR) Metrics

### 4.1 CFR by Federal Entity (State)

In [68]:
cfr_by_state = (
    df_unified.groupby("ENTIDAD_RES")["DECEASED"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .reset_index()
)
cfr_by_state.columns = ["ENTIDAD_RES", "CFR (%)"]
cfr_by_state["ENTIDAD_RES"] = pd.to_numeric(
    cfr_by_state["ENTIDAD_RES"], errors="coerce", downcast="integer"
)
cfr_by_state["State"] = cfr_by_state["ENTIDAD_RES"].map(STATE_MAPPING)
cfr_by_state = cfr_by_state[["State", "CFR (%)"]]

print("Case Fatality Rate (CFR) by State:")
print(cfr_by_state.round(2))

export_processed_csv(cfr_by_state, "cfr_by_state_final_for_map.csv")

Case Fatality Rate (CFR) by State:
                              State  CFR (%)
0                          GUERRERO     9.04
1                   BAJA CALIFORNIA     8.69
2                      QUINTANA ROO     8.46
3                        TAMAULIPAS     8.01
4               MICHOACÁN DE OCAMPO     7.63
5                           MORELOS     6.54
6                        NUEVO LEÓN     6.42
7                            COLIMA     6.34
8                   SAN LUIS POTOSÍ     5.77
9                           SINALOA     5.59
10                        CHIHUAHUA     5.42
11                           OAXACA     5.25
12                          NAYARIT     5.24
13                          JALISCO     5.15
14                       GUANAJUATO     5.15
15                           SONORA     5.07
16                          TABASCO     4.62
17  VERACRUZ DE IGNACIO DE LA LLAVE     4.50
18              BAJA CALIFORNIA SUR     4.44
19                        ZACATECAS     4.10
20                  

### 4.2 CFR by Age Cohort

In [69]:
cfr_by_age_group = (
    df_unified.groupby("AGE_GROUP")["DECEASED"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .reset_index()
)
cfr_by_age_group.columns = ["AGE_GROUP", "CFR (%)"]

print("Case Fatality Rate (CFR) by Age Group:")
print(cfr_by_age_group.round(2))

Case Fatality Rate (CFR) by Age Group:
  AGE_GROUP  CFR (%)
0       60+    25.07
1     36-59     5.82
2      0-17     1.33
3     18-35     0.97


### 4.3 Specific CFR by Comorbidity

In [70]:
fatality_results = {}

for comorbidity in COMORBIDITIES_FOR_ANALYSIS:
    df_filtered = df_unified[df_unified[comorbidity] == 1]
    fatality_results[comorbidity] = (
        df_filtered["DECEASED"].mean() * 100 if not df_filtered.empty else np.nan
    )

df_top_5 = (
    pd.DataFrame(fatality_results.items(), columns=["Comorbidity", "Specific CFR (%)"])
    .sort_values(by="Specific CFR (%)", ascending=False)
)

print("Top 5 comorbidities with the highest specific CFR:")
print(df_top_5.head(5).round(2))

export_processed_csv(df_top_5, "top5_comorbidities.csv")

Top 5 comorbidities with the highest specific CFR:
      Comorbidity  Specific CFR (%)
5   CHRONIC_RENAL             26.58
3            COPD             23.18
0        DIABETES             20.99
6  CARDIOVASCULAR             19.70
1    HYPERTENSION             18.46
Exported artifact: c:\Users\willi\Documents\3 cuatri Datos\DataProcessing_Proyect\data\processed\top5_comorbidities.csv


### 4.4 Intubation Proportion among Hospitalized Patients

In [71]:
df_hospitalized = df_unified[df_unified["PATIENT_TYPE"] == 2].copy()
total_hospitalized = len(df_hospitalized)
total_intubated = (df_hospitalized["INTUBATED"] == 1).sum()
intubation_ratio = (
    (total_intubated / total_hospitalized) * 100 if total_hospitalized > 0 else 0
)

df_intubation_analysis = pd.DataFrame(
    {
        "Metric": [
            "Total Hospitalized Patients",
            "Intubated Patients",
            "Intubation Ratio (%)",
        ],
        "Value": [total_hospitalized, total_intubated, intubation_ratio],
    }
)

print("Intubation analysis for hospitalized patients:")
print(df_intubation_analysis)

export_processed_csv(df_intubation_analysis, "intubation_comparison.csv")

Intubation analysis for hospitalized patients:
                        Metric          Value
0  Total Hospitalized Patients  200031.000000
1           Intubated Patients   33656.000000
2         Intubation Ratio (%)      16.825392
Exported artifact: c:\Users\willi\Documents\3 cuatri Datos\DataProcessing_Proyect\data\processed\intubation_comparison.csv


### 4.5 Time Series: Confirmed Cases, Deaths, and Daily CFR

In [72]:
df_unified["SYMPTOMS_ONSET_DATE"] = pd.to_datetime(
    df_unified["SYMPTOMS_ONSET_DATE"], errors="coerce"
)
df_unified["DEATH_DATE"] = pd.to_datetime(df_unified["DEATH_DATE"], errors="coerce")

confirmed_cases = df_unified[
    df_unified["FINAL_COVID_CLASSIFICATION"].isin([1, 2, 3])
].copy()

cases_series = (
    confirmed_cases.groupby("SYMPTOMS_ONSET_DATE")
    .size()
    .reset_index(name="Confirmed_Cases")
    .rename(columns={"SYMPTOMS_ONSET_DATE": "Date"})
)

deaths_df = confirmed_cases[confirmed_cases["DEATH_DATE"].notna()].copy()
deaths_series = (
    deaths_df.groupby("DEATH_DATE")
    .size()
    .reset_index(name="Deaths")
    .rename(columns={"DEATH_DATE": "Date"})
)

final_time_series = (
    pd.merge(cases_series, deaths_series, on="Date", how="outer")
    .sort_values("Date")
    .fillna(0)
)
final_time_series["Confirmed_Cases"] = final_time_series["Confirmed_Cases"].astype(int)
final_time_series["Deaths"] = final_time_series["Deaths"].astype(int)
final_time_series["Daily_CFR (%)"] = np.where(
    final_time_series["Confirmed_Cases"] > 0,
    (final_time_series["Deaths"] / final_time_series["Confirmed_Cases"]) * 100,
    0.0,
)

print("Time series preview:")
print(final_time_series.tail().round(2))

export_processed_csv(final_time_series, "cases_deaths_time_series.csv")

Time series preview:
          Date  Confirmed_Cases  Deaths  Daily_CFR (%)
312 2025-11-09                5       0            0.0
313 2025-11-10                3       0            0.0
314 2025-11-11                4       0            0.0
315 2025-11-12                2       0            0.0
316 2025-11-13                3       0            0.0
Exported artifact: c:\Users\willi\Documents\3 cuatri Datos\DataProcessing_Proyect\data\processed\cases_deaths_time_series.csv


### 4.6 CFR by Cumulative Comorbidity Count

In [73]:
df_count = df_unified[["DECEASED"] + COMORBIDITIES_FOR_ANALYSIS].copy()
df_count["NUM_COMORBIDITIES"] = df_count[COMORBIDITIES_FOR_ANALYSIS].eq(1).sum(axis=1)
df_unified["NUM_COMORBIDITIES"] = df_count["NUM_COMORBIDITIES"]

comorb_bins = [-0.5, 0.5, 1.5, 2.5, df_unified["NUM_COMORBIDITIES"].max() + 1]
comorb_labels = ["0 comorbidities", "1 comorbidity", "2 comorbidities", "3+ comorbidities"]

df_unified["COMORBIDITY_GROUP"] = pd.cut(
    df_unified["NUM_COMORBIDITIES"],
    bins=comorb_bins,
    labels=comorb_labels,
    right=False,
    include_lowest=True,
)

cfr_by_comorbidity_count = (
    df_unified.groupby("COMORBIDITY_GROUP")["DECEASED"]
    .mean()
    .mul(100)
    .reset_index(name="CFR (%)")
)
cfr_by_comorbidity_count.columns = ["Comorbidity Group", "CFR (%)"]

print("Case Fatality Rate (CFR) by Number of Comorbidities:")
print(cfr_by_comorbidity_count.round(2))

export_processed_csv(cfr_by_comorbidity_count, "deaths_by_comorbidity_count.csv")

Case Fatality Rate (CFR) by Number of Comorbidities:
  Comorbidity Group  CFR (%)
0   0 comorbidities     3.30
1     1 comorbidity     8.51
2   2 comorbidities    15.90
3  3+ comorbidities    23.07
Exported artifact: c:\Users\willi\Documents\3 cuatri Datos\DataProcessing_Proyect\data\processed\deaths_by_comorbidity_count.csv


## 5. Master Dataset Export

In [74]:
export_processed_csv(df_unified, "cleaned_unified_mexico_covid_data.csv")

Exported artifact: c:\Users\willi\Documents\3 cuatri Datos\DataProcessing_Proyect\data\processed\cleaned_unified_mexico_covid_data.csv


# 6. Final Dataset Validation

In [75]:
print("DataFrame Preview:")
print(df_unified.head())

print("\nDataFrame Structure and Memory Usage:")
print(df_unified.info())

print("\nComplete List of Standardized Columns:")
for col in df_unified.columns:
    print(f"'{col}',")

DataFrame Preview:
  LAST_UPDATED REGISTRATION_ID  ORIGIN SECTOR ENTIDAD_UM  SEX ENTIDAD_NAC  \
0   2025-11-18         g771db5       1     15         21    1          21   
1   2025-11-18         g574d0a       1      4         09    1          09   
2   2025-11-18         g941a21       1     12         10    1          10   
3   2025-11-18         geaaed1       1      4         09    2          09   
4   2025-11-18         g9c2f41       1      5         24    2          24   

  ENTIDAD_RES MUNICIPIO_RES PATIENT_TYPE  ... FINAL_COVID_CLASSIFICATION  \
0          21           085            1  ...                        6.0   
1          09           017            1  ...                        7.0   
2          10           005            1  ...                        7.0   
3          09           006            2  ...                        7.0   
4          24           014            2  ...                        5.0   

  FINAL_FLU_CLASSIFICATION MIGRANT  COUNTRY_OF_NATIONALITY  C